# ERP Factorial — 03: Analysis

Extracts mean amplitudes, parses conditions into factors, runs a repeated-measures
ANOVA and planned pairwise contrasts with FDR correction.

Exports a long-format CSV compatible with R, JASP, jamovi, and SPSS.

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import (
    load_config,
    extract_mean_amplitudes,
    run_paired_tests,
    run_anova,
)
import pandas as pd

# ── Update these paths ──
cfg     = load_config('../../../configs/your_experiment.yaml')
cfg_erp = load_config('../../../configs/your_erp_factorial.yaml')

print("Setup OK")

In [ ]:
# ── ROIs and time windows committed from notebook 02 ──
rois = {
    'left_roi':  ['Ch1', 'Ch2', 'Ch3'],
    'right_roi': ['Ch4', 'Ch5', 'Ch6'],
}

time_windows = {
    'early': [0.10, 0.25],
    'late':  [0.25, 0.50],
}

# ── Extract all granular conditions (conditions=None extracts everything) ──
df_raw = extract_mean_amplitudes(
    cfg,
    window_name='your_window',
    rois=rois,
    time_windows=time_windows,
    conditions=None,
    save_csv=False,
    verbose=True,
)

# ── Collapse granular conditions into factor-level groups ──
# ── Update groupings to match your design ──
groupings = {
    'factor1_level1_left':  ['factor1_level1_pos1', 'factor1_level1_pos2'],
    'factor1_level1_right': ['factor1_level1_pos3', 'factor1_level1_pos4'],
    'factor1_level2_left':  ['factor1_level2_pos1', 'factor1_level2_pos2'],
    'factor1_level2_right': ['factor1_level2_pos3', 'factor1_level2_pos4'],
}

rows = []
for grp_name, components in groupings.items():
    sub = df_raw[df_raw['condition'].isin(components)]
    collapsed = sub.groupby(['subject', 'window', 'roi', 'channels',
                             'time_window', 'tmin', 'tmax'], as_index=False).agg(
        mean_amp_uv=('mean_amp_uv', 'mean')
    )
    collapsed['condition'] = grp_name
    rows.append(collapsed)

df = pd.concat(rows, ignore_index=True)
print(f"Collapsed: {df['condition'].nunique()} conditions, {df['subject'].nunique()} subjects")

In [ ]:
# ── Parse condition names into factors ──
# ── Update the parsing logic to match your condition naming convention ──
# Example: 'factor1_level1_left' → factor1='level1', side='left'
df['factor1'] = df['condition'].apply(lambda x: x.split('_')[1])  # adapt as needed
df['side']    = df['condition'].apply(lambda x: 'left' if x.endswith('left') else 'right')

print("Factor levels:")
print(f"   factor1: {df['factor1'].unique().tolist()}")
print(f"   side:    {df['side'].unique().tolist()}")

In [ ]:
# ── Repeated-measures ANOVA ──
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests

anova_rows = []
for (roi, tw), group in df.groupby(['roi', 'time_window']):
    aov = AnovaRM(
        group, depvar='mean_amp_uv', subject='subject',
        within=['factor1', 'side']   # ── update factors ──
    ).fit()
    for effect, row in aov.anova_table.iterrows():
        anova_rows.append({
            'roi': roi, 'time_window': tw, 'effect': effect,
            'F': row['F Value'], 'df1': row['Num DF'],
            'df2': row['Den DF'], 'p': row['Pr > F'],
        })

anova_df = pd.DataFrame(anova_rows)
_, anova_df['p_fdr'], _, _ = multipletests(anova_df['p'], method='fdr_bh')
anova_df['sig_fdr'] = anova_df['p_fdr'] < 0.05

print(f"FDR correction across {len(anova_df)} tests")
print(anova_df[anova_df['sig_fdr']].to_string(index=False) or "No significant effects.")

In [ ]:
# ── Planned pairwise contrasts ──
# ── Update contrasts to match your design hypotheses ──
contrasts = [
    ('factor1_level1_left', 'factor1_level1_right'),
    ('factor1_level2_left', 'factor1_level2_right'),
]

results = run_paired_tests(
    df,
    contrasts=contrasts,
    cfg=cfg, window_name='your_window',
    save_csv=True, verbose=True,
)

In [ ]:
# ── Export long-format CSV for external software (R, JASP, jamovi, SPSS) ──
from eeg_toolkit.evoked import _get_erp_dir

out_path = _get_erp_dir(cfg) / "stats" / "factorial_amplitudes_for_export.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")